In [ ]:
import numpy as np
import torch
from torch.nn import *
from glob import glob
from subprocess import Popen
class Add(Sequential):
    def forward(s, x):
        return sum(s(x)for s in s)
class F(Module):
    def __init__(s, f, *_):
        super().__init__()
        s.f = f
        s._ = _
    def forward(s, x):
        return s.f(x, *s._)
def kl_div(mu, logvar):
    return -.5*(1+logvar-mu**2-logvar.exp()).sum()
A = np.array([i.split(b'|')for i in Popen('sqlite3;cards.cdb;select id,alias from datas'.split(';'),stdout=-1).stdout.read().split()],'I')
A = A[A[:,1]!=0]
A = [np.where((_:=A[A[:,0].searchsorted(i).clip(max=len(A)-1)])[:,0]==i,_[:,1],i)for i in glob('deck/*/*')for i in[np.array([i for i in open(i).read().split('!')[0].split()if i.isdigit()],'I')]if 55<=len(i)<=75]
try:
    I,_,o = torch.load('deck2.pk',weights_only=False)
    m = Sequential(
        Embedding(len(I), 256, scale_grad_by_freq=True),
        F(lambda _:(torch.stack([_.cumsum(1),_.flip(1).cumsum(1).flip(1)],1)/torch.arange(1,_.size(1)+1)[:,None]).flatten(1,2)),
        SiLU(True),
        Linear(256, len(I)),
        LogSoftmax(2)
    )
    m.load_state_dict(_)
except FileNotFoundError:
    I = np.unique(np.concat(A))
    m = Sequential(
        Embedding(len(I), 256, scale_grad_by_freq=True),
        F(lambda _:(torch.stack([_.cumsum(1),_.flip(1).cumsum(1).flip(1)],1)/torch.arange(1,_.size(1)+1)[:,None]).flatten(1,2)),
        SiLU(True),
        Linear(256, len(I)),
        LogSoftmax(2)
    )
    o = torch.optim.Adam(m.parameters(), .01)
A = [torch.from_numpy(I.searchsorted(i).astype('h'))for i in A]

In [ ]:
while True:
    L = 0
    for _ in range(1000):
        o.zero_grad()
        a = [A[i].long()for i in torch.randint(len(A),[16])]
        x = torch.stack([i[torch.randperm(len(i))[:min(map(len,a))]]for i in a])
        y = m(x)
        t = torch.stack([torch.zeros(len(I)).put_(i,torch.tensor(1/len(i)).expand(i.shape),True)for i in a])
        l = functional.kl_div(y, t[:,None])
        l.backward()
        o.step()
        L += l.data
    torch.save((I,m.state_dict(),o),'deck2.pk')
    print(L)